In [0]:
%python
customers_initial_df = spark.read.format("delta").load("/mnt/data/bronze/customers")
products_initial_df = spark.read.format("delta").load("/mnt/data/bronze/products")
orders_initial_df = spark.read.format("delta").load("/mnt/data/bronze/orders")

In [0]:
from pyspark.sql import Window
from pyspark.sql.types import DecimalType
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType

def prepare_customer_dimension(source_df):

    # Text Standardization: trim + uppercase for all string columns
    string_cols = [f.name for f in source_df.schema.fields if f.dataType.simpleString() == "string"]

    df = source_df
    for col in string_cols:
        df = df.withColumn(col, F.upper(F.trim(F.col(col))))

    # Address Concatenation (null-safe)
    df = df.withColumn(
        "fullAddress", 
        F.concat_ws(" | ", F.col("address"), F.col("city"), F.col("state"), F.col("postalCode"))
    )

    # Customer Segmentation
    df = df.withColumn(
        "customerSegment",
        F.concat_ws("_", F.col("region"), F.col("customerTier"))
    )

    # Date Processing  
    df = df.withColumn("onboardDate", F.to_date(F.col("onboardDate"), "yyyy-MM-dd")) \
           .withColumn("customerAge", F.datediff(F.current_date(), F.col("onboardDate")))

    # Surrogate Key Generation (row_number)
    window_spec = Window.orderBy("customerId")
    df = df.withColumn("customerSK", F.row_number().over(window_spec))

    # Metadata Addition
    df = df.withColumn("processedAt", F.current_timestamp())

    return df


In [0]:
def prepare_product_dimension(source_df):

    #trim and uppercase all string columns
    string_cols = [f.name for f in source_df.schema.fields if f.dataType.simpleString() == "string"]
    df = source_df
    for col in string_cols:
        df = df.withColumn(col, F.upper(F.trim(F.col(col))))

    # Convert empty strings to nulls
    df = df.replace("", None)

    # Conversion — numeric fields with safe cast
    numeric_cols = ["unitCost", "listPrice"]

    for col in numeric_cols:
        df = df.withColumn(
            col,
            F.when(F.col(col).rlike("^[0-9]*\\.?[0-9]+$"), F.col(col).cast(DecimalType(10, 2)))
             .otherwise(F.lit(0.0))
        )

    # Product Classification
    df = df.withColumn(
        "productFamily",
        F.concat_ws("_", F.col("category"), F.col("productCode"))
    )

    # Gross Margin Calculation
    df = df.withColumn(
        "grossMarginPct",
        F.when(F.col("listPrice") != 0,
               ((F.col("listPrice") - F.col("unitCost")) / F.col("listPrice")) * 100
        ).otherwise(0)
    )

    # Surrogate Key — productId
    window_spec = Window.orderBy("productId")
    df = df.withColumn("productSK", F.row_number().over(window_spec))

    # Metadata & date conversion
    df = df.withColumn("launchDate", F.to_date("launchDate", "yyyy-MM-dd")) \
           .withColumn("processedAt", F.current_timestamp())

    return df


In [0]:


def build_fact_order_line(orders_df, customer_dim_df, product_dim_df):
    
    # Standardize text → trim + uppercase for ALL string columns
    string_cols = [c for c, t in orders_df.dtypes if t == "string"]
    df = orders_df
    for col in string_cols:
        df = df.withColumn(col, F.upper(F.trim(F.col(col))))

    # Convert empty strings to nulls
    df = df.replace("", None)

    # Convert numeric string columns to numeric safely
    df = df.withColumn("orderedQty", F.col("orderedQty").cast(IntegerType())) \
           .withColumn("unitPrice", 
               F.when(F.col("unitPrice").rlike("^[0-9]*\\.?[0-9]+$"), 
                      F.col("unitPrice").cast(DecimalType(10, 2)))
                .otherwise(0.0)
           ) \
           .withColumn("discountPct", 
               F.when(F.col("discountPct").rlike("^[0-9]*\\.?[0-9]+$"),
                      F.col("discountPct").cast(DecimalType(5, 2)))
                .otherwise(0.0)
           )

    # Convert dates
    df = df.withColumn("orderDate", F.to_date("orderDate", "yyyy-MM-dd")) \
           .withColumn("shipDate", F.to_date("shipDate", "yyyy-MM-dd"))

    # Join with Customer Dimension
    df = df.join(
        customer_dim_df.select("customerId", "customerSK", "customerSegment"),
        on="customerId",
        how="left"
    )

    # Join with Product Dimension
    df = df.join(
        product_dim_df.select("productId", "productSK", "unitCost"),
        on="productId",
        how="left"
    )

    # Fact Calculations
    df = df.withColumn("lineSubtotal", F.col("orderedQty") * F.col("unitPrice"))

    df = df.withColumn("discountAmount",
            F.col("lineSubtotal") * (F.col("discountPct") / 100)
        )

    df = df.withColumn("lineTotal", F.col("lineSubtotal") - F.col("discountAmount"))

    df = df.withColumn("lineCost", F.col("orderedQty") * F.col("unitCost"))

    df = df.withColumn("lineMargin", F.col("lineTotal") - F.col("lineCost"))

    # Add metadata
    df = df.withColumn("processedAt", F.current_timestamp())

    # Select final fact schema
    fact_df = df.select(
        "orderId", "orderLineId", "orderDate", "shipDate",
        "customerSK", "productSK","productId",
        "orderedQty", "unitPrice", "discountPct",
        "lineSubtotal", "discountAmount", "lineTotal",
        "lineCost", "lineMargin",
        "currency", "orderStatus", "salesChannel",
        "salesRepId", "shippingMethod",
        "customerSegment",
        "processedAt"
    )

    return fact_df

In [0]:
dim_customer = prepare_customer_dimension(customers_initial_df)
dim_product = prepare_product_dimension(products_initial_df)
fact_order_line = build_fact_order_line(orders_initial_df,dim_customer,dim_product)

display(dim_customer)
display(dim_product)
display(fact_order_line)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customerId,customerName,industry,region,customerTier,isActive,email,phone,address,city,state,postalCode,onboardDate,creditRating,fullAddress,customerSegment,customerAge,customerSK,processedAt
101,ACME CORP,MANUFACTURING,NORTH AMERICA,ENTERPRISE,true,JOHN.DOE@ACME.COM,+1-555-0101,123 MAIN ST,NEW YORK,NY,10001,2020-01-15,A,123 MAIN ST | NEW YORK | NY | 10001,NORTH AMERICA_ENTERPRISE,2116,1,2025-10-31T05:00:41.380Z
102,BRIGHT RETAIL,RETAIL,NORTH AMERICA,MID-MARKET,true,CONTACT@BRIGHTRETAIL.COM,555.0102,456 OAK AVE,CHICAGO,IL,60601,2019-03-22,A,456 OAK AVE | CHICAGO | IL | 60601,NORTH AMERICA_MID-MARKET,2415,2,2025-10-31T05:00:41.380Z
103,CITY HOSPITAL,HEALTHCARE,NORTH AMERICA,ENTERPRISE,true,ADMIN@CITYHOSPITAL.ORG,(555) 0103,789 HEALTH BLVD,BOSTON,MA,02101,2021-07-10,A,789 HEALTH BLVD | BOSTON | MA | 02101,NORTH AMERICA_ENTERPRISE,1574,3,2025-10-31T05:00:41.380Z
104,DELTA AIRLINES,TRANSPORTATION,NORTH AMERICA,STRATEGIC,true,INFO@DELTA.COM,1-555-0104,1001 AIRPORT WAY,ATLANTA,GA,30301,2018-11-05,A,1001 AIRPORT WAY | ATLANTA | GA | 30301,NORTH AMERICA_STRATEGIC,2552,4,2025-10-31T05:00:41.380Z
105,FRONTIER EDUCATION,EDUCATION,NORTH AMERICA,MID-MARKET,true,HELLO@FRONTIER.EDU,555-0105,2020 CAMPUS DR,AUSTIN,TX,73301,2022-02-28,A,2020 CAMPUS DR | AUSTIN | TX | 73301,NORTH AMERICA_MID-MARKET,1341,5,2025-10-31T05:00:41.380Z
106,GAMMA STORES,RETAIL,LATIN AMERICA,SMB,true,VENTAS@GAMMA.COM,+52-555-0106,AV. PRINCIPAL 100,MEXICO CITY,DF,01000,2020-09-12,B,AV. PRINCIPAL 100 | MEXICO CITY | DF | 01000,LATIN AMERICA_SMB,1875,6,2025-10-31T05:00:41.380Z
107,HELIOS ENERGY,ENERGY,EUROPE,STRATEGIC,true,CONTACT@HELIOS.EU,+44-20-5550107,10 ENERGY PLAZA,LONDON,,SW1A 1AA,2019-06-18,A,10 ENERGY PLAZA | LONDON | | SW1A 1AA,EUROPE_STRATEGIC,2327,7,2025-10-31T05:00:41.380Z
108,INNOVA LABS,TECHNOLOGY,EUROPE,MID-MARKET,true,INFO@INNOVA.DE,+49-30-5550108,TECH STR. 42,BERLIN,BE,10115,2021-12-03,A,TECH STR. 42 | BERLIN | BE | 10115,EUROPE_MID-MARKET,1428,8,2025-10-31T05:00:41.380Z


productId,productName,category,unitCost,listPrice,status,launchDate,productCode,screenSize,processor,productFamily,grossMarginPct,productSK,processedAt
1001,PHOTON LAPTOP 14,COMPUTING,865.5,1299.0,ACTIVE,2023-01-15,LT,15.6,INTEL I7,COMPUTING_LT,33.37182448036952,1,2025-10-31T05:00:42.916Z
1002,LUMEN MONITOR 27,ACCESSORIES,205.0,329.0,ACTIVE,2023-02-20,MN,27.0,4K DISPLAY,ACCESSORIES_MN,37.68996960486322,2,2025-10-31T05:00:42.916Z
1003,NIMBUS ROUTER PRO,NETWORKING,102.5,189.99,ACTIVE,2023-03-10,RT,null,WIFI 6,NETWORKING_RT,46.04979209432076,3,2025-10-31T05:00:42.916Z
1004,AURORA TABLET 11,COMPUTING,455.0,749.0,ACTIVE,2023-04-05,TB,11.0,ARM PROCESSOR,COMPUTING_TB,39.25233644859813,4,2025-10-31T05:00:42.916Z
1005,SOLARDOCK PRO,ACCESSORIES,140.25,229.0,DISCONTINUED,2022-12-01,DK,0,USB-C HUB,ACCESSORIES_DK,38.75545851528384,5,2025-10-31T05:00:42.916Z
1006,ION SERVER BLADE,COMPUTING,682.75,1125.0,ACTIVE,2023-05-18,SV,1U,XEON GOLD,COMPUTING_SV,39.31111111111111,6,2025-10-31T05:00:42.916Z


orderId,orderLineId,orderDate,shipDate,customerSK,productSK,productId,orderedQty,unitPrice,discountPct,lineSubtotal,discountAmount,lineTotal,lineCost,lineMargin,currency,orderStatus,salesChannel,salesRepId,shippingMethod,customerSegment,processedAt
7001,1,2024-01-10,2024-01-12,1,1,1001,4,1299.0,10.5,5196.0,545.5799999999999,4650.42,3462.0,1188.42,USD,SHIPPED,FIELD_SALES,REP001,STANDARD,NORTH AMERICA_ENTERPRISE,2025-10-31T05:00:44.814Z
7001,2,2024-01-10,2024-01-13,1,2,1002,2,329.0,5.0,658.0,32.9,625.1,410.0,215.10000000000002,USD,SHIPPED,FIELD_SALES,REP001,STANDARD,NORTH AMERICA_ENTERPRISE,2025-10-31T05:00:44.814Z
7002,1,2024-01-12,2024-01-14,2,2,1002,3,329.0,15.0,987.0,148.04999999999998,838.95,615.0,223.95000000000005,USD,SHIPPED,PARTNER,PART01,EXPRESS,NORTH AMERICA_MID-MARKET,2025-10-31T05:00:44.814Z
7003,1,2024-01-14,2024-01-16,3,3,1003,5,189.99,0.0,949.95,0.0,949.95,512.5,437.45000000000005,USD,SHIPPED,ONLINE,null,STANDARD,NORTH AMERICA_ENTERPRISE,2025-10-31T05:00:44.814Z
7004,1,2024-01-18,2024-01-20,4,4,1004,1,749.0,25.5,749.0,190.995,558.005,455.0,103.005,USD,SHIPPED,ONLINE,WEB001,EXPRESS,NORTH AMERICA_STRATEGIC,2025-10-31T05:00:44.814Z
7005,1,2024-01-20,2024-01-24,5,1,1001,2,1299.0,12.0,2598.0,311.76,2286.24,1731.0,555.2399999999998,USD,SHIPPED,FIELD_SALES,REP002,STANDARD,NORTH AMERICA_MID-MARKET,2025-10-31T05:00:44.814Z
7006,1,2024-01-22,null,6,4,1004,3,749.0,0.0,2247.0,0.0,2247.0,1365.0,882.0,USD,PROCESSING,ONLINE,WEB002,STANDARD,LATIN AMERICA_SMB,2025-10-31T05:00:44.814Z
7007,1,2024-01-25,2024-01-28,7,5,1005,6,229.0,8.5,1374.0,116.79,1257.21,841.5,415.71000000000004,EUR,SHIPPED,PARTNER,PART02,STANDARD,EUROPE_STRATEGIC,2025-10-31T05:00:44.814Z
7008,1,2024-01-27,null,8,6,1006,2,1125.0,20.0,2250.0,450.0,1800.0,1365.5,434.5,USD,PENDING,FIELD_SALES,REP003,EXPRESS,EUROPE_MID-MARKET,2025-10-31T05:00:44.814Z
7009,1,2024-01-29,2024-02-01,5,3,1003,4,189.99,7.5,759.96,56.997,702.9630000000001,410.0,292.9630000000001,USD,SHIPPED,ONLINE,WEB001,STANDARD,NORTH AMERICA_MID-MARKET,2025-10-31T05:00:44.814Z


In [0]:
customers_write = "/mnt/data/silver/customers"

(
    dim_customer
    .write
    .format("delta")
    .mode("overwrite")   # Bronze is overwritten with raw ingestion
    .option("overwriteSchema", "true")
    .save(customers_write)
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
products_write = "/mnt/data/silver/products"

(
    dim_product
    .write
    .format("delta")
    .mode("overwrite")   # Bronze is overwritten with raw ingestion
    .option("overwriteSchema", "true")
    .save(products_write)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
orders_write = "/mnt/data/silver/orders"

(
    fact_order_line.write
    .format("delta")
    .mode("overwrite")   # Bronze is overwritten with raw ingestion
    .option("overwriteSchema", "true")
    .save(orders_write)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
